In [1]:

import pandas as pd
import numpy as np

# Customers table
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "customer_name": [
        "Ahmad", "Sara", "Omar",
        "Lina", "Khaled", "Noor"
    ],
    "city": [
        "Amman", "Irbid", "Amman",
        "Zarqa", "Irbid", "Amman"
    ]
})

# Orders table
orders = pd.DataFrame({
    "order_id": [
        101, 102, 103, 104, 105, 106,
        107, 108, 109, 110, 111, 103
    ],
    "customer_id": [
        1, 2, 1, 3, 4, 2,
        5, 6, 3, 5, 1, 1
    ],
    "category": [
        "Electronics", "Clothes", "Books",
        "Electronics", "Clothes", "Books",
        "Electronics", "Clothes", "Books",
        "Clothes", "Electronics", "Books"
    ],
    "amount": [
        250, 80, 35, 500, -20, 45,
        300, np.nan, 60, 110, 200, 35
    ],
    "status": [
        "Completed", "Completed", "Completed",
        "Completed", "Completed", "Cancelled",
        "Completed", "Completed", "Completed",
        "Completed", "Cancelled", "Completed"
    ]
})

print("Customers Shape:", customers.shape)
print("Orders Shape:", orders.shape)

print("\nCustomers:")
print(customers)

print("\nOrders:")
print(orders)


Customers Shape: (6, 3)
Orders Shape: (12, 5)

Customers:
   customer_id customer_name   city
0            1         Ahmad  Amman
1            2          Sara  Irbid
2            3          Omar  Amman
3            4          Lina  Zarqa
4            5        Khaled  Irbid
5            6          Noor  Amman

Orders:
    order_id  customer_id     category  amount     status
0        101            1  Electronics   250.0  Completed
1        102            2      Clothes    80.0  Completed
2        103            1        Books    35.0  Completed
3        104            3  Electronics   500.0  Completed
4        105            4      Clothes   -20.0  Completed
5        106            2        Books    45.0  Cancelled
6        107            5  Electronics   300.0  Completed
7        108            6      Clothes     NaN  Completed
8        109            3        Books    60.0  Completed
9        110            5      Clothes   110.0  Completed
10       111            1  Electronics   20

In [6]:

# 1. Check missing values
print("Missing Values:")
print(orders.isnull().sum())

# 2. Check negative amounts
print("\nNegative Amounts:")
print(orders[orders["amount"] < 0])

# 3. Check duplicate order IDs
print("\nDuplicate Orders:")
print(orders[orders.duplicated(
    subset=["order_id"],
    keep=False
)])

# 4. Check order status
print("\nOrder Status:")
print(orders["status"].value_counts())


Missing Values:
order_id       0
customer_id    0
category       0
amount         1
status         0
dtype: int64

Negative Amounts:
   order_id  customer_id category  amount     status
4       105            4  Clothes   -20.0  Completed

Duplicate Orders:
    order_id  customer_id category  amount     status
2        103            1    Books    35.0  Completed
11       103            1    Books    35.0  Completed

Order Status:
status
Completed    10
Cancelled     2
Name: count, dtype: int64


In [7]:

# Step 1: Create a copy
clean_orders = orders.copy()

# Step 2: Remove duplicate orders
clean_orders = clean_orders.drop_duplicates(
    subset=["order_id"]
)

# Step 3: Keep completed orders
clean_orders = clean_orders[
    clean_orders["status"] == "Completed"
].copy()

# Step 4: Keep valid positive amounts
clean_orders = clean_orders[
    clean_orders["amount"].notna()
    & (clean_orders["amount"] > 0)
].copy()

# Step 5: Check the cleaned data
print("Original Orders:", orders.shape)
print("Clean Orders:", clean_orders.shape)

print("\nCleaned Dataset:")
print(clean_orders)


Original Orders: (12, 5)
Clean Orders: (7, 5)

Cleaned Dataset:
   order_id  customer_id     category  amount     status
0       101            1  Electronics   250.0  Completed
1       102            2      Clothes    80.0  Completed
2       103            1        Books    35.0  Completed
3       104            3  Electronics   500.0  Completed
6       107            5  Electronics   300.0  Completed
8       109            3        Books    60.0  Completed
9       110            5      Clothes   110.0  Completed


In [8]:

# Calculate total sales
total_sales = clean_orders["amount"].sum()

print("Total Sales:", total_sales)


Total Sales: 1335.0


In [9]:

# Calculate sales by category
sales_by_category = clean_orders.groupby("category")["amount"].sum()

print(sales_by_category)


category
Books            95.0
Clothes         190.0
Electronics    1050.0
Name: amount, dtype: float64


In [10]:

# Join orders with customers
merged_data = pd.merge(
    clean_orders,
    customers,
    on="customer_id",
    how="inner",
    validate="many_to_one"
)

print("Merged Shape:", merged_data.shape)

print(merged_data[
    ["order_id", "customer_name", "city", "amount"]
])


Merged Shape: (7, 7)
   order_id customer_name   city  amount
0       101         Ahmad  Amman   250.0
1       102          Sara  Irbid    80.0
2       103         Ahmad  Amman    35.0
3       104          Omar  Amman   500.0
4       107        Khaled  Irbid   300.0
5       109          Omar  Amman    60.0
6       110        Khaled  Irbid   110.0


In [11]:

# Calculate total sales by city
sales_by_city = merged_data.groupby("city")["amount"].sum()

# Sort from highest to lowest
sales_by_city = sales_by_city.sort_values(ascending=False)

print("Sales by City:")
print(sales_by_city)


Sales by City:
city
Amman    845.0
Irbid    490.0
Name: amount, dtype: float64


In [12]:

import sqlite3

# Create an in-memory database
conn = sqlite3.connect(":memory:")

# Save DataFrames as SQL tables
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

clean_orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")


Database created successfully!


In [13]:

query = """
SELECT *
FROM orders;
"""

result = pd.read_sql_query(query, conn)

print(result)


   order_id  customer_id     category  amount     status
0       101            1  Electronics   250.0  Completed
1       102            2      Clothes    80.0  Completed
2       103            1        Books    35.0  Completed
3       104            3  Electronics   500.0  Completed
4       107            5  Electronics   300.0  Completed
5       109            3        Books    60.0  Completed
6       110            5      Clothes   110.0  Completed


In [14]:

query = """
SELECT order_id, category, amount
FROM orders
WHERE amount > 100;
"""

result = pd.read_sql_query(query, conn)

print(result)


   order_id     category  amount
0       101  Electronics   250.0
1       104  Electronics   500.0
2       107  Electronics   300.0
3       110      Clothes   110.0


In [15]:

query = """
SELECT
    o.order_id,
    c.customer_name,
    c.city,
    o.amount
FROM orders AS o
INNER JOIN customers AS c
ON o.customer_id = c.customer_id
ORDER BY o.order_id;
"""

result = pd.read_sql_query(query, conn)

print(result)


   order_id customer_name   city  amount
0       101         Ahmad  Amman   250.0
1       102          Sara  Irbid    80.0
2       103         Ahmad  Amman    35.0
3       104          Omar  Amman   500.0
4       107        Khaled  Irbid   300.0
5       109          Omar  Amman    60.0
6       110        Khaled  Irbid   110.0


In [16]:

query = """
SELECT
    c.city,
    COUNT(o.order_id) AS total_orders,
    SUM(o.amount) AS total_sales
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.city
ORDER BY total_sales DESC;
"""

result = pd.read_sql_query(query, conn)

print(result)


    city  total_orders  total_sales
0  Amman             4        845.0
1  Irbid             3        490.0


In [19]:

query = """
SELECT
    o.category,
    COUNT(o.order_id) AS total_orders,
    SUM(o.amount) AS total_sales
FROM orders AS o
GROUP BY o.category
ORDER BY total_sales DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)


      category  total_orders  total_sales
0  Electronics             3       1050.0
1      Clothes             2        190.0
2        Books             2         95.0


    city  total_orders  total_sales
0  Amman             4        845.0
1  Irbid             3        490.0
